##### Download and read the EHR Dataset

In [4]:
pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
df = pd.read_csv("dataset/synthetic_ehr_dataset.csv")  # adjust to the actual filename
print(df.head())

   Patient ID Sex  Age  Visit Date               Diagnosis ICD-10 Code  \
0       75722   F   78  2022-11-10            Hypertension         E11   
1       80185   M   72  2015-07-13       Diabetes Mellitus         I25   
2       19865   M   51  2024-01-16                  Asthma         F32   
3       76700   M   41  2022-11-16          Osteoarthritis         E11   
4       92992   M   75  2017-11-26  Chronic Kidney Disease         I25   

           Treatment     Comorbidity  Systolic BP  Diastolic BP  Heart Rate  \
0           Dialysis         Obesity          110            73          74   
1            Insulin         Obesity          173            90         119   
2     ACE inhibitors  Hyperlipidemia          138           103          77   
3  Lifestyle changes  Hyperlipidemia          129            88         116   
4      Beta-blockers     Sleep Apnea          124            64         120   

   Temperature  Glucose  Cholesterol Clinical Outcome  Relapse Risk  
0         

##### Generate a new dataset for fine-tuning in the format : [ input prompt : deterministic output ]
In this specific case we will create a dataset with following format
Diagnostic feature1 .... featureN -> clinical outcome -> STABLE | IMPROVED | WORSENED

In [6]:
pip install scikit-learn

  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 37.3 MB/s eta 0:00:00a 0:00:01
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 72.3 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

df = pd.read_csv("dataset/synthetic_ehr_dataset.csv")

# cleaning dataset
df["patient_sex"] = df["Sex"].map({"M": "Male", "F": "Female"})
df["patient_age"] = df["Age"]
df["diagnosis"] = df["Diagnosis"]
df["icd_code"] = df["ICD-10 Code"]
df["treatment"] = df["Treatment"]
df["comorbidity"] = df["Comorbidity"]
df["systolic_bp"] = df["Systolic BP"]
df["diastolic_bp"] = df["Diastolic BP"]
df["heart_rate"] = df["Heart Rate"]
df["temperature"] = df["Temperature"]
df["glucose"] = df["Glucose"]
df["cholesterol"] = df["Cholesterol"]
df["clinical_outcome"] = df["Clinical Outcome"]

columns_to_keep = [
    "patient_sex", "patient_age", "diagnosis", "icd_code", "treatment",
    "comorbidity", "systolic_bp", "diastolic_bp", "heart_rate",
    "temperature", "glucose", "cholesterol", "clinical_outcome"
]
df_ready = df[columns_to_keep]

# --- Split into train/test 80/20 randomly on clinical_outcome as that is going to be our inference---
train_df, test_df = train_test_split(
    df_ready, test_size=0.2, random_state=42, stratify=df_ready["clinical_outcome"]
)

# Create output directory
os.makedirs("dataset/splits", exist_ok=True)

# Save as CSVs
train_df.to_csv("dataset/splits/train.csv", index=False)
test_df.to_csv("dataset/splits/test.csv", index=False)

print("✅ Done!")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print("Columns:", list(train_df.columns))
print("Sample record:\n", train_df.iloc[0])

✅ Done!
Train: 8000 | Test: 2000
Columns: ['patient_sex', 'patient_age', 'diagnosis', 'icd_code', 'treatment', 'comorbidity', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'temperature', 'glucose', 'cholesterol', 'clinical_outcome']
Sample record:
 patient_sex                   Male
patient_age                     59
diagnosis                     COPD
icd_code                       E11
treatment           ACE inhibitors
comorbidity         Hyperlipidemia
systolic_bp                    170
diastolic_bp                    74
heart_rate                      81
temperature                   39.3
glucose                        115
cholesterol                    138
clinical_outcome          Worsened
Name: 4749, dtype: object
